# NLI · news · topic assignment

Hits `/nli` and applies **taxonomy-aware topic resolution**: scores every node in the hierarchy, then for each branch above threshold picks the most specific supported label — unless multiple siblings fire, in which case it returns their common parent (evidence spread = broader topic).

**Traversal rule (per root):**
1. Root score < threshold → branch skipped entirely
2. Root above threshold, 0 children above threshold → assign root
3. Root above threshold, 1 child above threshold → recurse on that child
4. Root above threshold, 2+ children above threshold → assign root (evidence too broad)

Config and taxonomy live in `config/topics.yaml`. Test cases live in `eval/fixtures/nli-news-topics.json`.

In [ ]:
import os, json, requests, yaml
from pathlib import Path

NLP_BASE_URL = os.environ.get('NLP_BASE_URL', 'https://wiig.dia.fi.upm.es/nlp').rstrip('/')
ENDPOINT     = f'{NLP_BASE_URL}/nli'
HEADERS      = {'Content-Type': 'application/json', 'X-API-Key': f'{os.environ.get("NLP_API_KEY")}'}
TIMEOUT      = 120

cfg = yaml.safe_load(Path('../config/topics.yaml').read_text(encoding='utf-8'))
TAXONOMY            = cfg['taxonomy']
SCORE_THRESHOLD     = cfg['score_threshold']
HYPOTHESIS_TEMPLATE = cfg['topic_hypothesis_template']

def _all_labels(taxonomy):
    labels = []
    for node in taxonomy:
        labels.append(node['label'])
        for child in node.get('children', []):
            labels.append(child['label'])
            for grandchild in child.get('children', []):
                labels.append(grandchild['label'])
    return list(dict.fromkeys(labels))

ALL_LABELS = _all_labels(TAXONOMY)

CASES = json.loads(Path('fixtures/nli-news-topics.json').read_text(encoding='utf-8'))
print(f'{len(CASES)} cases -> {ENDPOINT}')
print(f'{len(ALL_LABELS)} labels scored per article  threshold={SCORE_THRESHOLD}')
print('taxonomy roots:', [n['label'] for n in TAXONOMY])

In [ ]:
def build_premise(inp):
    headline = (inp.get('headline') or '').strip()
    text     = (inp.get('text') or '').strip()
    return f'{headline}. {text}' if headline else text


def call_nli(premise, hypotheses):
    body = {'text': premise, 'hypotheses': hypotheses, 'hypothesis_template': HYPOTHESIS_TEMPLATE}
    r = requests.post(ENDPOINT, json=body, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json().get('scores', [])


def _resolve_node(node, scores, threshold):
    if scores.get(node['label'], 0) < threshold:
        return None
    children_above = [c for c in node.get('children', []) if scores.get(c['label'], 0) >= threshold]
    if len(children_above) == 0:
        return node['label']                                      # most specific: this node
    elif len(children_above) == 1:
        return _resolve_node(children_above[0], scores, threshold)  # recurse deeper
    else:
        return node['label']                                      # multiple children → parent


def assign_topics(inp):
    raw    = call_nli(build_premise(inp), ALL_LABELS)
    scores = {ALL_LABELS[i]: item['score'] for i, item in enumerate(raw)}
    resolved = [_resolve_node(root, scores, SCORE_THRESHOLD) for root in TAXONOMY]
    return [t for t in resolved if t is not None], scores

In [ ]:
def render_taxonomy(scores, resolved, threshold):
    for root in TAXONOMY:
        r_score = scores.get(root['label'], 0)
        marker  = '\u25b6' if root['label'] in resolved else ' '
        dim     = '' if r_score >= threshold else '  \u00b7'
        print(f'  {marker} {root["label"]:30s}  {r_score:.3f}{dim}')
        for child in root.get('children', []):
            c_score  = scores.get(child['label'], 0)
            c_marker = '\u25b6' if child['label'] in resolved else ' '
            c_dim    = '' if c_score >= threshold else '  \u00b7'
            print(f'      {c_marker} {child["label"]:26s}  {c_score:.3f}{c_dim}')

In [ ]:
results = []
for case in CASES:
    print(f"── {case['id']}: {case['description']}")
    try:
        resolved, scores = assign_topics(case['input'])
    except Exception as exc:
        print('   ERROR:', exc); print()
        results.append((case['id'], False))
        continue

    render_taxonomy(scores, resolved, SCORE_THRESHOLD)
    print(f'  Resolved: {resolved if resolved else "(none)"}')

    expected = case['expected']['topics']
    if not expected:
        passed = len(resolved) == 0
        note   = f'  expected none, got {resolved}' if not passed else ''
    else:
        missing = [t for t in expected if t not in resolved]
        passed  = len(missing) == 0
        note    = f'  missing: {missing}' if missing else ''

    print(f'  {"\u2705" if passed else "\u274c"} {case["id"]}{note}')
    print()
    results.append((case['id'], passed))

In [ ]:
passed_n = sum(1 for _, p in results if p)
print('SCORECARD \u00b7 /nli  (taxonomy-aware topic assignment)')
print(f'  Cases passing: {passed_n}/{len(results)}')
for cid, p in results:
    print(f"   {'\u2705' if p else '\u274c'} {cid}")